# Lesson 2a: Dynamic Programming Theory

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand Generalized Policy Iteration** - The fundamental framework underlying DP
2. **Prove Convergence Theorems** - Mathematical guarantees for policy iteration and value iteration
3. **Analyze Computational Complexity** - Time and space requirements of DP algorithms
4. **Study DP Variants** - Asynchronous, prioritized, and modified approaches
5. **Explore Optimality Principles** - Bellman optimality and contraction mappings
6. **Compare DP Methods** - Trade-offs between different dynamic programming approaches

This notebook provides the theoretical foundation for dynamic programming in reinforcement learning, building upon the MDP theory from Lesson 1a.

---

## Table of Contents

1. [Introduction to Dynamic Programming](#intro)
2. [Generalized Policy Iteration](#gpi)
3. [Convergence Theory](#convergence)
4. [Bellman Operators and Contraction](#contraction)
5. [Policy Iteration Analysis](#policy-iteration)
6. [Value Iteration Analysis](#value-iteration)
7. [Dynamic Programming Variants](#variants)
8. [Computational Complexity](#complexity)
9. [Limitations and Extensions](#limitations)

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Callable
from dataclasses import dataclass

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. Introduction to Dynamic Programming <a id='intro'></a>

### What is Dynamic Programming?

**Dynamic Programming (DP)** is a method for solving complex problems by breaking them down into simpler subproblems. In reinforcement learning, DP refers to a collection of algorithms that can be used to compute optimal policies given a perfect model of the environment as a Markov Decision Process (MDP).

### Key Properties of DP

DP algorithms share two key properties:

1. **Optimal Substructure**: The optimal solution can be decomposed into optimal solutions of subproblems
2. **Overlapping Subproblems**: The same subproblems are solved multiple times

### Why Dynamic Programming for RL?

The Bellman equations provide recursive decompositions that DP algorithms exploit:

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma V^\pi(s')]$$

$$V^*(s) = \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma V^*(s')]$$

These equations express the value of a state in terms of values of successor states, exhibiting the **optimal substructure** property.

### DP Requirements

DP algorithms require:
- **Full model knowledge**: Access to $p(s',r|s,a)$ for all $s, a, s', r$
- **Finite state/action spaces**: For exact computation
- **Markov property**: Future is independent of past given present

### DP Algorithm Classes

Two main classes:

1. **Prediction** (Policy Evaluation): Compute $V^\pi$ for a given policy $\pi$
2. **Control** (Finding Optimal Policy): Compute $V^*$ and $\pi^*$

Control algorithms include:
- Policy Iteration
- Value Iteration
- Modified Policy Iteration
- Asynchronous DP

## 2. Generalized Policy Iteration <a id='gpi'></a>

### The GPI Framework

**Generalized Policy Iteration (GPI)** is the fundamental framework underlying almost all reinforcement learning methods. GPI consists of two interacting processes:

1. **Policy Evaluation**: Making the value function consistent with the current policy
2. **Policy Improvement**: Making the policy greedy with respect to the current value function

```
     π₀ → V^π₀ → π₁ → V^π₁ → π₂ → ... → π* → V*
     
     E: Evaluation
     I: Improvement
     
     π₀ --E--> V^π₀ --I--> π₁ --E--> V^π₁ --I--> π₂ --E--> ...
```

### Mathematical Formulation

**Policy Evaluation Step**:
$$V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s'} p(s'|s,a)[r(s,a,s') + \gamma V_k(s')]$$

**Policy Improvement Step**:
$$\pi_{k+1}(s) = \arg\max_a \sum_{s'} p(s'|s,a)[r(s,a,s') + \gamma V^{\pi_k}(s')]$$

### GPI Convergence Intuition

The two processes interact in a way that drives both the policy and value function toward optimality:

- **Evaluation**: Makes value function accurate for current policy
- **Improvement**: Makes policy better with respect to value function
- **Stabilization**: When both processes stabilize, we have found $\pi^*$ and $V^*$

### GPI Diagram

```
V
│
│   ╱ 
│  ╱ ← greedy
│ ╱   policy
│╱
└─────────── π
 ╲
  ╲ ← policy
   ╲  evaluation
    ╲
```

The value function and policy pull in different directions, but together they find the optimal solution.

### Theorem 2.1: GPI Convergence

**Theorem**: If both policy evaluation and policy improvement stabilize, then the policy and value function must be optimal.

**Proof**:

Suppose both processes have stabilized. Then:

1. **From evaluation stabilization**:
   $$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} p(s'|s,a)[r + \gamma V^\pi(s')]$$
   
   This is the Bellman expectation equation.

2. **From improvement stabilization**:
   $$\pi(s) = \arg\max_a \sum_{s'} p(s'|s,a)[r + \gamma V^\pi(s')]$$
   
   The policy is greedy with respect to its own value function.

3. **Combining both**:
   $$V^\pi(s) = \max_a \sum_{s'} p(s'|s,a)[r + \gamma V^\pi(s')]$$
   
   This is the Bellman optimality equation, which has a unique solution $V^*$.

Therefore, $V^\pi = V^*$ and $\pi = \pi^*$. $\square$

## 3. Convergence Theory <a id='convergence'></a>

### Policy Improvement Theorem

**Theorem 3.1 (Policy Improvement)**:

Let $\pi$ and $\pi'$ be deterministic policies such that for all $s \in \mathcal{S}$:

$$Q^\pi(s, \pi'(s)) \geq V^\pi(s)$$

Then $V^{\pi'}(s) \geq V^\pi(s)$ for all $s \in \mathcal{S}$.

If the inequality is strict for any state, then $V^{\pi'}(s) > V^\pi(s)$ for at least one state.

**Proof**:

Starting from the assumption:

$$V^\pi(s) \leq Q^\pi(s, \pi'(s))$$

$$= \mathbb{E}[R_{t+1} + \gamma V^\pi(S_{t+1}) | S_t = s, A_t = \pi'(s)]$$

$$= \mathbb{E}_{\pi'}[R_{t+1} + \gamma V^\pi(S_{t+1}) | S_t = s]$$

Applying the inequality again to $V^\pi(S_{t+1})$:

$$\leq \mathbb{E}_{\pi'}[R_{t+1} + \gamma Q^\pi(S_{t+1}, \pi'(S_{t+1})) | S_t = s]$$

$$= \mathbb{E}_{\pi'}[R_{t+1} + \gamma \mathbb{E}[R_{t+2} + \gamma V^\pi(S_{t+2}) | S_{t+1}] | S_t = s]$$

$$= \mathbb{E}_{\pi'}[R_{t+1} + \gamma R_{t+2} + \gamma^2 V^\pi(S_{t+2}) | S_t = s]$$

Continuing this process:

$$\leq \mathbb{E}_{\pi'}[R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ... | S_t = s]$$

$$= V^{\pi'}(s)$$

Therefore, $V^{\pi'}(s) \geq V^\pi(s)$ for all states. $\square$

### Corollary: Greedy Policy Improvement

If $\pi'$ is the greedy policy with respect to $V^\pi$:

$$\pi'(s) = \arg\max_a Q^\pi(s,a)$$

Then:

$$Q^\pi(s, \pi'(s)) = \max_a Q^\pi(s,a) \geq Q^\pi(s, \pi(s)) = V^\pi(s)$$

Therefore, $\pi'$ is at least as good as $\pi$.

### Monotonic Improvement

**Theorem 3.2**: Policy iteration produces a sequence of policies $\pi_0, \pi_1, \pi_2, ...$ such that:

$$V^{\pi_0} \leq V^{\pi_1} \leq V^{\pi_2} \leq ... \leq V^*$$

where the inequality is element-wise over all states.

**Proof**: Direct application of the Policy Improvement Theorem at each iteration. $\square$

### Finite Convergence

**Theorem 3.3**: For a finite MDP, policy iteration converges to the optimal policy in a finite number of iterations.

**Proof**:

1. There are only finitely many deterministic policies: $|\mathcal{A}|^{|\mathcal{S}|}$

2. Each policy iteration step either:
   - Strictly improves the policy (moves to a new, better policy), OR
   - The policy is already optimal (no change)

3. Since we cannot return to a previous policy (values are monotonically increasing), and there are finitely many policies, the algorithm must terminate.

4. At termination, the policy is greedy with respect to its own value function, satisfying the Bellman optimality equation.

Therefore, policy iteration converges in at most $|\mathcal{A}|^{|\mathcal{S}|}$ iterations. $\square$

In practice, convergence is much faster, often in just a few iterations.

## 4. Bellman Operators and Contraction <a id='contraction'></a>

### Bellman Expectation Operator

Define the **Bellman expectation operator** $T^\pi$ for policy $\pi$:

$$(T^\pi V)(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma V(s')]$$

Policy evaluation finds the fixed point: $V^\pi = T^\pi V^\pi$

### Bellman Optimality Operator

Define the **Bellman optimality operator** $T^*$:

$$(T^* V)(s) = \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma V(s')]$$

Value iteration finds the fixed point: $V^* = T^* V^*$

### Contraction Mapping Theorem

**Definition**: An operator $T$ is a **$\gamma$-contraction** in the max-norm if:

$$\|T V_1 - T V_2\|_\infty \leq \gamma \|V_1 - V_2\|_\infty$$

for all value functions $V_1, V_2$, where $\|V\|_\infty = \max_s |V(s)|$.

**Theorem 4.1 (Banach Fixed Point Theorem)**: 

If $T$ is a $\gamma$-contraction with $\gamma < 1$ in a complete metric space, then:

1. $T$ has a unique fixed point $V^*$: $T V^* = V^*$
2. Repeated application converges to $V^*$: $\lim_{k \to \infty} T^k V = V^*$ for any $V$
3. Convergence is exponential: $\|T^k V - V^*\|_\infty \leq \gamma^k \|V - V^*\|_\infty$

### Theorem 4.2: Bellman Operators are Contractions

Both $T^\pi$ and $T^*$ are $\gamma$-contractions.

**Proof for $T^*$**:

Let $V_1, V_2$ be arbitrary value functions. For any state $s$:

$$|(T^* V_1)(s) - (T^* V_2)(s)|$$

$$= \left| \max_a \sum_{s'} p(s'|s,a)[r + \gamma V_1(s')] - \max_a \sum_{s'} p(s'|s,a)[r + \gamma V_2(s')] \right|$$

Using the property $|\max_a f(a) - \max_a g(a)| \leq \max_a |f(a) - g(a)|$:

$$\leq \max_a \left| \sum_{s'} p(s'|s,a) \gamma [V_1(s') - V_2(s')] \right|$$

$$\leq \max_a \sum_{s'} p(s'|s,a) \gamma |V_1(s') - V_2(s')|$$

$$\leq \max_a \sum_{s'} p(s'|s,a) \gamma \|V_1 - V_2\|_\infty$$

$$= \gamma \|V_1 - V_2\|_\infty \max_a \sum_{s'} p(s'|s,a)$$

$$= \gamma \|V_1 - V_2\|_\infty$$

Since this holds for all states:

$$\|T^* V_1 - T^* V_2\|_\infty \leq \gamma \|V_1 - V_2\|_\infty$$

Therefore, $T^*$ is a $\gamma$-contraction. $\square$

### Convergence Rate

From the contraction property:

$$\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$$

**Key insight**: Error decreases exponentially at rate $\gamma$. This explains why:
- Higher discount factors ($\gamma$ close to 1) converge slower
- Lower discount factors converge faster but are more myopic

### Example: Convergence Rate Calculation

In [ ]:
def compute_convergence_iterations(gamma: float, initial_error: float, threshold: float) -> int:
    """
    Compute number of iterations needed to reach threshold.
    
    From: ||V_k - V*|| ≤ γ^k ||V_0 - V*||
    We need: γ^k * initial_error ≤ threshold
    So: k ≥ log(threshold/initial_error) / log(γ)
    """
    if gamma >= 1 or gamma <= 0:
        raise ValueError("Gamma must be in (0, 1)")
    
    k = np.log(threshold / initial_error) / np.log(gamma)
    return int(np.ceil(k))


# Example calculations
print("Convergence Iterations Analysis")
print("=" * 60)
print(f"Initial error: 1.0, Target threshold: 1e-6\n")

for gamma in [0.5, 0.9, 0.95, 0.99, 0.999]:
    iters = compute_convergence_iterations(gamma, 1.0, 1e-6)
    print(f"γ = {gamma:5.3f}: {iters:4d} iterations")

print("\nObservation: Higher γ → slower convergence")
print("✅ Convergence rate demonstration complete")

### Visualization of Contraction

In [ ]:
# Visualize how error decreases with different gamma values
gammas = [0.5, 0.7, 0.9, 0.99]
iterations = np.arange(0, 50)

fig, ax = plt.subplots(figsize=(10, 6))

for gamma in gammas:
    errors = gamma ** iterations
    ax.plot(iterations, errors, label=f'γ = {gamma}', linewidth=2, marker='o', markersize=4, alpha=0.7)

ax.set_xlabel('Iteration k', fontsize=12)
ax.set_ylabel('Error Bound: γ^k', fontsize=12)
ax.set_title('Contraction Mapping: Exponential Error Decay', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=1e-6, color='red', linestyle='--', label='Typical threshold (1e-6)', alpha=0.5)

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/2a_contraction.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Contraction visualization created")

## 5. Policy Iteration Analysis <a id='policy-iteration'></a>

### Algorithm Review

**Policy Iteration Algorithm**:

```
1. Initialize π arbitrarily
2. Repeat:
     a. Policy Evaluation: Solve V^π = T^π V^π
     b. Policy Improvement: π' ← greedy(V^π)
     c. If π' = π, stop
3. Return π
```

### Computational Complexity per Iteration

**Policy Evaluation** (solving linear system):
- Direct solution: $O(|\mathcal{S}|^3)$ using matrix inversion
- Iterative solution: $O(|\mathcal{S}|^2 |\mathcal{A}|)$ per sweep

**Policy Improvement**:
- Computing action values: $O(|\mathcal{S}|^2 |\mathcal{A}|)$
- Selecting greedy actions: $O(|\mathcal{S}| |\mathcal{A}|)$

### Number of Iterations

**Theorem 5.1**: Policy iteration converges in at most $|\mathcal{A}|^{|\mathcal{S}|}$ policy iterations.

**Practical observation**: Typically converges in far fewer iterations, often $O(|\mathcal{S}|)$ or even $O(\log |\mathcal{S}|)$ iterations.

### Why Policy Iteration Works Well

1. **Large policy improvements**: Each step can improve many states simultaneously
2. **Exact evaluation**: Knowing $V^\pi$ exactly leads to optimal improvement
3. **No backtracking**: Monotonic improvement guarantees forward progress

### Stopping Criterion

Policy iteration stops when:
$$\pi_{k+1}(s) = \pi_k(s) \text{ for all } s \in \mathcal{S}$$

This guarantees optimality because:
$$\pi_k = \arg\max_a Q^{\pi_k}(s,a) \implies V^{\pi_k} = V^*$$

## 6. Value Iteration Analysis <a id='value-iteration'></a>

### Algorithm Review

**Value Iteration Algorithm**:

```
1. Initialize V arbitrarily
2. Repeat:
     V(s) ← max_a Σ_{s'} p(s'|s,a)[r + γV(s')]
   Until ||V_{k+1} - V_k|| < θ
3. π(s) ← argmax_a Σ_{s'} p(s'|s,a)[r + γV(s')]
4. Return π
```

### Computational Complexity per Iteration

**Value Update**:
- For each state: $O(|\mathcal{A}| |\mathcal{S}|)$
- Total per iteration: $O(|\mathcal{S}|^2 |\mathcal{A}|)$

**Policy Extraction** (done once at end):
- $O(|\mathcal{S}|^2 |\mathcal{A}|)$

### Convergence Guarantee

**Theorem 6.1**: Value iteration converges to $V^*$ as $k \to \infty$.

**Proof**: Direct application of Banach fixed point theorem, since $T^*$ is a contraction. $\square$

### Convergence Rate

From contraction property:
$$\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$$

### Stopping Criterion

Common stopping conditions:

1. **Value change threshold**:
   $$\|V_{k+1} - V_k\|_\infty < \theta$$

2. **Guaranteed optimality**:
   $$\|V_k - V^*\|_\infty \leq \frac{\theta (1 - \gamma)}{2\gamma}$$
   
   This follows from:
   $$\|V_{k+1} - V_k\| < \theta \implies \|V_k - V^*\| < \frac{\theta}{1 - \gamma}$$

### Value Iteration as Truncated Policy Iteration

Value iteration can be viewed as policy iteration with:
- Policy evaluation truncated to **one sweep**
- Policy improvement after each sweep

This explains why value iteration often requires more iterations than policy iteration but is faster per iteration.

### Comparison: Policy Iteration vs Value Iteration

| Aspect | Policy Iteration | Value Iteration |
|--------|------------------|----------------|
| Iterations | Fewer ($k$ small) | More ($k$ large) |
| Per-iteration cost | Higher (solve linear system) | Lower (one sweep) |
| Total time | Often faster | Variable |
| Intermediate policies | Available | Not explicitly tracked |
| Convergence | Finite steps | Asymptotic |
| When optimal | $\pi_k = \pi_{k-1}$ | $\|V_k - V_{k-1}\| < \theta$ |

## 7. Dynamic Programming Variants <a id='variants'></a>

### 7.1 Asynchronous Dynamic Programming

**Idea**: Update states in any order, potentially skipping states or updating some more than others.

**Asynchronous Value Iteration**:
```
Repeat:
  Select state s (in any order)
  V(s) ← max_a Σ_{s'} p(s'|s,a)[r + γV(s')]
```

**Theorem 7.1**: Asynchronous value iteration converges to $V^*$ if all states continue to be updated.

**Advantages**:
- More flexible scheduling
- Can focus computation on important states
- Better for online/real-time applications

### 7.2 Prioritized Sweeping

**Idea**: Update states in priority order based on potential value change.

**Priority Definition**:
$$p(s) = \left| V(s) - \max_a \sum_{s'} p(s'|s,a)[r + \gamma V(s')] \right|$$

This is the magnitude of the Bellman error.

**Algorithm**:
```
1. Initialize priority queue Q
2. For all states s, compute p(s) and add to Q
3. Repeat:
     s ← Q.pop() (state with highest priority)
     Update V(s)
     For each predecessor s' of s:
       Recompute p(s')
       Update Q with new priority
```

**Advantages**:
- Focuses on states that need updating most
- Can dramatically reduce iterations
- Especially effective in large state spaces

### 7.3 Modified Policy Iteration

**Idea**: Truncate policy evaluation to $k$ steps instead of full convergence.

**Algorithm**:
```
1. Initialize π
2. Repeat:
     Perform k sweeps of policy evaluation
     π ← greedy(V)
```

**Special Cases**:
- $k = 1$: Value iteration
- $k = \infty$: Standard policy iteration
- $k \in (1, \infty)$: Trade-off between the two

**Optimal $k$**:
- Problem-dependent
- Empirically: $k \in [3, 10]$ often works well

### 7.4 Real-Time Dynamic Programming

**Idea**: Update only states visited by agent following current policy.

**Algorithm**:
```
Repeat (for each episode):
  s ← initial state
  Repeat (for each step):
    a ← greedy action from s
    Execute a, observe s'
    Update V(s)
    s ← s'
```

**Advantages**:
- Focuses on relevant states
- Works online during execution
- Useful when state space is huge but most states are irrelevant

### 7.5 In-Place Updates

**Standard (synchronous)**:
```python
V_new = np.zeros_like(V)
for s in states:
    V_new[s] = max_a ...
V = V_new
```

**In-place (asynchronous)**:
```python
for s in states:
    V[s] = max_a ...  # Uses updated values immediately
```

**Benefits**:
- Halves memory requirement
- Often converges faster
- Updated values propagate immediately

## 8. Computational Complexity <a id='complexity'></a>

### Time Complexity Summary

For an MDP with $|\mathcal{S}|$ states and $|\mathcal{A}|$ actions:

| Algorithm | Per Iteration | Total Iterations | Total Complexity |
|-----------|---------------|------------------|------------------|
| Policy Evaluation (iterative) | $O(\|\mathcal{S}\|^2 \|\mathcal{A}\|)$ | $O(\frac{1}{1-\gamma} \log \frac{1}{\theta})$ | $O(\frac{\|\mathcal{S}\|^2 \|\mathcal{A}\|}{1-\gamma} \log \frac{1}{\theta})$ |
| Policy Evaluation (direct) | $O(\|\mathcal{S}\|^3)$ | 1 | $O(\|\mathcal{S}\|^3)$ |
| Policy Iteration | $O(\|\mathcal{S}\|^3)$ | $O(\|\mathcal{A}\|^{\|\mathcal{S}\|})$ worst, $O(\|\mathcal{S}\|)$ typical | Variable |
| Value Iteration | $O(\|\mathcal{S}\|^2 \|\mathcal{A}\|)$ | $O(\frac{1}{1-\gamma} \log \frac{1}{\theta})$ | $O(\frac{\|\mathcal{S}\|^2 \|\mathcal{A}\|}{1-\gamma} \log \frac{1}{\theta})$ |

### Space Complexity

**Model Storage**:
- Transition probabilities $P$: $O(|\mathcal{S}|^2 |\mathcal{A}|)$
- Rewards $R$: $O(|\mathcal{S}|^2 |\mathcal{A}|)$ or $O(|\mathcal{S}| |\mathcal{A}|)$ if state-action only

**Value Function Storage**:
- State values $V$: $O(|\mathcal{S}|)$
- Action values $Q$: $O(|\mathcal{S}| |\mathcal{A}|)$

**Total**: $O(|\mathcal{S}|^2 |\mathcal{A}|)$ dominated by model storage

### The Curse of Dimensionality

**Problem**: State space grows exponentially with number of state variables.

**Example**: Grid world with $n$ dimensions, $k$ positions per dimension:
$$|\mathcal{S}| = k^n$$

For $k=10, n=5$: $|\mathcal{S}| = 100,000$ states

**Consequences**:
- DP becomes impractical for $|\mathcal{S}| > 10^6$
- Memory requirements prohibitive
- Computation time excessive

**Solutions** (covered in later lessons):
- Function approximation
- State aggregation
- Sample-based methods (Monte Carlo, TD learning)
- Deep reinforcement learning

### Practical Scalability

DP is practical when:
- $|\mathcal{S}| \leq 10^6$ (millions of states)
- Sparse transitions (most $p(s'|s,a) = 0$)
- Asynchronous/prioritized updates used
- State space has structure to exploit

## 9. Limitations and Extensions <a id='limitations'></a>

### Limitations of Dynamic Programming

1. **Requires Full Model**:
   - Must know $p(s',r|s,a)$ for all transitions
   - Often unavailable in real-world problems
   - Solution: Model-free methods (Lessons 3-5)

2. **Computational Complexity**:
   - Scales poorly with state space size
   - Curse of dimensionality
   - Solution: Approximate DP, function approximation

3. **Assumes Markov Property**:
   - Future must be independent of past given present
   - Not always satisfied
   - Solution: State augmentation, POMDP methods

4. **Discrete State/Action Spaces**:
   - Exact DP requires discrete, finite spaces
   - Many problems are continuous
   - Solution: Discretization, continuous-space methods

### Extensions and Advanced Topics

#### 9.1 Approximate Dynamic Programming

Use function approximation for value function:
$$V(s) \approx \hat{V}(s; \mathbf{w})$$

Examples:
- Linear: $\hat{V}(s; \mathbf{w}) = \mathbf{w}^T \phi(s)$
- Neural network: $\hat{V}(s; \mathbf{w}) = \text{NN}(s; \mathbf{w})$

#### 9.2 Partially Observable MDPs (POMDPs)

Agent doesn't observe full state, only observations:
$$o_t \sim O(s_t)$$

Must maintain belief state:
$$b(s) = P(S_t = s | o_1, ..., o_t, a_1, ..., a_{t-1})$$

#### 9.3 Continuous State/Action Spaces

Methods:
- Discretization
- Linear-quadratic regulators (LQR)
- Fitted value iteration
- Policy gradient methods

#### 9.4 Multi-Agent MDPs

Multiple agents with coupled dynamics:
- Markov games
- Nash equilibria
- Cooperative vs competitive

#### 9.5 Hierarchical RL

Decompose problem into subtasks:
- Options framework
- HAM (Hierarchical Abstract Machines)
- MAXQ decomposition

### When to Use DP

**Use DP when**:
- ✅ Full model is available
- ✅ State space is manageable ($< 10^6$ states)
- ✅ Exact solution is needed
- ✅ Planning/simulation environment available

**Avoid DP when**:
- ❌ Model is unknown or uncertain
- ❌ State space is too large
- ❌ Real-time decisions needed
- ❌ Continuous state/action spaces

### Bridging to Model-Free Methods

DP provides foundation for understanding:
- **Monte Carlo methods**: Replace expectation with samples
- **Temporal Difference learning**: Bootstrap like DP, sample like MC
- **Q-learning**: Model-free value iteration
- **SARSA**: Model-free policy iteration

These connections will be explored in upcoming lessons.

---

## Summary

In this notebook, you have learned:

1. ✅ **Generalized Policy Iteration**: The fundamental framework for RL algorithms
2. ✅ **Convergence Theory**: Mathematical guarantees for DP methods
3. ✅ **Contraction Mappings**: Why Bellman operators converge exponentially
4. ✅ **Policy Iteration**: Finite convergence to optimal policy
5. ✅ **Value Iteration**: Asymptotic convergence with exponential rate
6. ✅ **DP Variants**: Asynchronous, prioritized, and modified approaches
7. ✅ **Complexity Analysis**: Time/space requirements and scalability
8. ✅ **Limitations**: When DP fails and what comes next

### Key Theoretical Results

1. **Policy Improvement Theorem**: Greedy policies are monotonically better
2. **Bellman Operators are Contractions**: Guarantees convergence at rate $\gamma$
3. **Finite Convergence of PI**: Optimal policy in $\leq |\mathcal{A}|^{|\mathcal{S}|}$ steps
4. **Exponential Convergence of VI**: Error bound $\|V_k - V^*\| \leq \gamma^k \|V_0 - V^*\|$

### Practical Insights

- **GPI Framework**: Understanding evaluation-improvement interaction is key
- **Trade-offs**: Policy iteration (fewer, expensive iterations) vs value iteration (many, cheap iterations)
- **Asynchronous methods**: Greater flexibility often improves performance
- **Curse of dimensionality**: DP practical only for moderate state spaces

### Next Steps

**Lesson 2b** will implement:
- Asynchronous value iteration
- Prioritized sweeping
- Modified policy iteration
- Complexity benchmarks
- Comparisons on larger problems

---

**Lesson 2a Complete!** 🎉

You now have a rigorous theoretical foundation for dynamic programming in reinforcement learning.